#### Setup

In [1]:
import shutil, subprocess, sys
from pathlib import Path

from dvc_dat.ledger import Ledger, Unlocked, bump

scratch_path = Path("scratch/q-triples").resolve()
shutil.rmtree(scratch_path, ignore_errors=True)   # a re-run starts clean
sys.modules.pop("demo_game", None)

# Quick: triples

The ledger is a quad store: every fact is a row
`(rev, subject, predicate, object)`, every write is one
transaction, and every read is a view of one revision. The quick
tour: write, fold, one and many, `[]`, the triple reads, then the
ledger's own settings.

## A ledger and a transaction

In [2]:
L = Ledger(scratch_path / "ledger", create=True)

with L.transaction(by="human:dan", why="season opener") as t:
    t.add("game/G1", "home", "G4")
    t.add("game/G1", "away", "G5")
    t.add("game/G1", "court", "north")
rev1 = t.rev
print(rev1.id, rev1.get("game/G1", "court"))   # 1 north

1 north


`by` is required; `why` and any other keyword are rows on the
transaction's own subject. The `with` block commits one revision.

## A second transaction, and folding

A read folds the rows at or before its revision into the state
they mean: on an undeclared predicate the latest add wins.

In [3]:
with L.transaction(by="human:dan") as t:
    t.add("game/G1", "court", "south")
rev2 = t.rev
print(rev2.get("game/G1", "court"))   # south: the latest add
print(rev1.get("game/G1", "court"))   # north: rev1 never changes
for r in rev2.rows("game/G1", "court"):
    print(r)                          # both rows are still stored
#    1 + game/G1   court   north
#    2 + game/G1   court   south

south
north
   1 + game/G1              court                    north
   2 + game/G1              court                    south


## One and many

A predicate is `one` unless declared `many`. Declare `officials`
many, then write both kinds in one transaction: the `one`
predicate replaces, the `many` one accumulates.

In [4]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/officials", "db/cardinality", "many")
    t.add("game/G1", "court", "east")       # one: replaces south
    t.add("game/G1", "officials", "kim")    # many: accumulates
    t.add("game/G1", "officials", "ortiz")
rev3 = t.rev
print(rev3.get("game/G1", "court"))         # east
print(rev3.all("game/G1", "officials"))     # ['kim', 'ortiz']
print(rev1.get("game/G1", "court"),
      rev1.all("game/G1", "officials"))     # north []: looking back

east
['kim', 'ortiz']
north []


The declaration is itself a row, on `db/pred/officials`. Any
earlier revision still reads as it did: `L.at(n)` is revision `n`.

## The `[]` accessor

One level is the subject's record; two levels is one predicate.
Writing mirrors reading, and a record write takes a list for a
`many` key.

In [5]:
with L.transaction(by="human:dan", lock=10) as t:      # the record write reads: lock=
    t["game/G1", "court"] = "west"                       # two levels
    t["game/G2"] = {"home": "G1", "away": "G3",          # one level
                    "officials": ["lee", "park"]}
rev = t.rev
print(rev["game/G1", "court"])   # west
print(rev["game/G2"])
# {'home': 'G1', 'away': 'G3', 'officials': ['lee', 'park']}

west
{'home': 'G1', 'away': 'G3', 'officials': ['lee', 'park']}


`rev[s, p]` on a `many` predicate raises and names `all`; a
missing fact is a `KeyError`.

## Triple reads

In [6]:
print(rev.get("game/G1", "score", default="tbd"))   # tbd: no raise
print(("game/G1", "home", "G4") in rev)             # True
print(rev.all("game/G1", "officials"))              # ['kim', 'ortiz']
print(len(rev.rows("game/G1", "court")))            # 4: the history

tbd
True
['kim', 'ortiz']
4


`get` answers a default instead of raising, `in` tests one triple,
`all` lists what a pattern leaves open, and `rows` is the unfolded
history.

## Declaring an index

In [7]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/home", "db/index", "true")

An index is storage only: queries on `home` answer the same,
faster.

## Naming a revision

In [8]:
with L.transaction(by="human:dan") as t:
    t["env/prod", "rev"] = str(rev2.id)
prod_rev = L.at(L.at()["env/prod", "rev"])
print(prod_rev.id, prod_rev.get("game/G1", "court"))   # 2 south

2 south


A name is any subject's `rev` row; `L.at` takes the id it holds.

## Typing a predicate

In [9]:
with L.transaction(by="human:dan") as t:
    t.add("db/pred/date", "db/type", "str")
try:
    with L.transaction(by="human:dan") as t:
        t.add("game/G1", "date", {"y": 2026, "m": 10})
except TypeError as e:
    print("TypeError:", e)   # json under a str predicate: not written

TypeError: 'date' is declared db/type 'str'; 'game/G1' was given json:'{"m":10,"y":2026}'; nothing written


## Connecting to code

A `code/<repo>` subject is a version clock tied to a git checkout.
`bump` raises the version and writes the commit beside it, read
from the checkout of that repo this process imported.

In [10]:
demo_repo_path = scratch_path / "demo-game"
(demo_repo_path / "demo_game").mkdir(parents=True, exist_ok=True)
(demo_repo_path / "demo_game/__init__.py").write_text("")
git_cmd = ["git", "-c", "user.name=demo", "-c", "user.email=demo@example.com"]
for args in (["init", "-q"], ["add", "."], ["commit", "-q", "-m", "0.0.1"]):
    subprocess.run(git_cmd + args, cwd=demo_repo_path, check=True)
sys.path.insert(0, str(demo_repo_path))
import demo_game

with L.transaction(by="human:dan", lock=10) as t:
    bump(t, "code/demo-game")
print(L.at()["code/demo-game"])
# {'version': '0.0.1', 'commit': '<the demo commit>'}

{'version': '0.0.1', 'commit': '4a045de4700a9eeccc23ec74b8d3bfec2a4b18de'}


## Locking the ledger

In [11]:
with L.transaction(by="human:dan", lock=10) as t:
    print(bump(t, "code/demo-game"))   # 0.0.2

0.0.2


`bump` reads the version through `t.previous_rev`, then writes
the next one. `lock=10` holds the write lock for the whole body,
so no other writer slips between the read and the write; past 10
seconds the body is interrupted and nothing is written.

A body can read through `t.previous_rev` itself, alter, and write
back. Here one more official joins `game/G2`.

In [12]:
with L.transaction(by="human:dan", lock=10) as t:
    g2 = t.previous_rev["game/G2"]                 # the record as it stands now
    t["game/G2"] = {**g2, "officials": g2["officials"] + ["vance"]}   # one more, written back
print("officials:", t.rev.all("game/G2", "officials"))

officials: ['lee', 'park', 'vance']


`t.previous_rev` is the previous revision, the top as it stands
while the body holds the write lock: the write is built on current
data, and no other writer can commit between the read and the
write. Without `lock=` there is no well-defined previous revision.
Notice the ledger refuses the read.

In [13]:
try:
    with L.transaction(by="human:dan") as t:       # no lock=
        t.previous_rev["game/G2"]
except Unlocked as e:
    print("Unlocked:", e)

Unlocked: the transaction is not locked: previous_rev is the top only under lock=


`bump`, `t[s] = {...}` and `del` read through `t.previous_rev` and
take `lock=`; `t[s, p] = o` and `t.add` read nothing and need none.

In [14]:
with L.transaction(by="human:dan") as t:           # no lock=: nothing is read
    t["game/G2", "court"] = "east"                 # an add on a one predicate
    t.add("game/G2", "officials", "wu")            # an add on a many predicate
print("game/G2:", t.rev["game/G2"])

game/G2: {'home': 'G1', 'away': 'G3', 'officials': ['lee', 'park', 'vance', 'wu'], 'court': 'east'}
